In [100]:
import json
import string
import pandas as pd
import numpy as np
import torch
from sklearn.metrics.pairwise import cosine_similarity
from transformers import DistilBertModel, DistilBertTokenizer

In [116]:
DATA_FILE = "dummy_data_elective3.json"
similarity_threshold = 0.75

In [117]:
def load_data(file_path):
    with open(file_path, 'r', encoding='utf-8') as file:
        data = json.load(file)
    return pd.DataFrame(data["CO_DATA"]), pd.DataFrame(data["PO_DATA"])

In [118]:
co_data, po_data = load_data(DATA_FILE)

In [119]:
def preprocess_text(text):
    """Converts text to lowercase and removes punctuation."""
    text = text.lower()
    text = ''.join([char for char in text if char not in string.punctuation])
    return text

In [120]:
co_data['cleaned_CO_Description'] = co_data['CO_Description'].apply(preprocess_text)
po_data['cleaned_PO_Description'] = po_data['PO_Description'].apply(preprocess_text)

In [121]:
distilbert_tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
distilbert_model = DistilBertModel.from_pretrained('distilbert-base-uncased')

In [122]:
def generate_embeddings(text_list):
    """Generates BERT embeddings for a list of text descriptions."""
    encoded_input = distilbert_tokenizer(text_list, padding=True, truncation=True, return_tensors='pt', max_length=128)
    distilbert_model.to('cpu')
    with torch.no_grad():
        model_output = distilbert_model(**encoded_input)
    # Use the mean of the token embeddings as the sentence embedding
    embeddings = model_output.last_hidden_state.mean(dim=1)
    return embeddings.tolist()

In [123]:
po_data['po_embeddings'] = generate_embeddings(po_data['cleaned_PO_Description'].tolist())
co_data['co_embeddings'] = generate_embeddings(co_data['cleaned_CO_Description'].tolist())

In [124]:
po_embeddings_array = np.array(po_data['po_embeddings'].tolist())
co_embeddings_array = np.array(co_data['co_embeddings'].tolist())

In [125]:
similarity_matrix = cosine_similarity(po_embeddings_array, co_embeddings_array)

In [126]:
similarity_matrix

array([[0.8414181 , 0.87616628, 0.87816543],
       [0.82868022, 0.86368404, 0.89127838],
       [0.83029098, 0.86309434, 0.87744369],
       [0.84985455, 0.88750726, 0.9089748 ],
       [0.82622147, 0.86697351, 0.88113947],
       [0.82439979, 0.86176919, 0.88266991],
       [0.8585566 , 0.89185358, 0.90680728],
       [0.7955903 , 0.80508473, 0.80027378],
       [0.74615888, 0.80497737, 0.85107528],
       [0.84144784, 0.86590016, 0.83859454],
       [0.80295764, 0.80379266, 0.83282839],
       [0.75880577, 0.81140807, 0.8072918 ],
       [0.89391459, 0.88975433, 0.82655457],
       [0.79669829, 0.83178419, 0.78422526],
       [0.59343359, 0.69298478, 0.69928919]])

In [127]:
# Create a DataFrame to store the relationships
relationships_df = pd.DataFrame(index=co_data['CO'], columns=po_data['PO'])

In [128]:
for i in range(similarity_matrix.shape[0]): # Repeat through rows (POs)
    for j in range(similarity_matrix.shape[1]): # Repeat through columns (COs)
        po = po_data.loc[i, 'PO']
        co = co_data.loc[j, 'CO']
        # Check if similarity is above the threshold and store as 1 or 0
        relationships_df.loc[co, po] = 1 if similarity_matrix[i, j] >= similarity_threshold else 0

In [129]:
relationships_df

PO,PO-a,PO-b,PO-c,PO-d,PO-e,PO-f,PO-g,PO-h,PO-i,PO-j,PO-k,PO-l,PO-m,PO-n,PO-o
CO,,,,,,,,,,,,,,,
CO1,1,1,1,1,1,1,1,1,0,1,1,1,1,1,0
CO2,1,1,1,1,1,1,1,1,1,1,1,1,1,1,0
CO3,1,1,1,1,1,1,1,1,1,1,1,1,1,1,0
